# Housing Prices Regression Analysis & EDA
This notebook demonstrates a complete end-to-end data analysis workflow, including environment setup, dataset inspection, target cleaning, preprocessing, and exploratory visualizations.

In [ ]:

from google.colab import files

uploaded = files.upload()



### 1.1 Python Libraries Import
We import essential packages for numerical computing, data structures, and plotting.

Python libraries
# New Section

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")



### 2.1 Load Raw Dataset
We read the uploaded housing prices CSV into a pandas DataFrame for analysis.

In [ ]:

df = pd.read_csv('housing_prices_regression.csv')

print(df)



### 2.2 Preview First 5 Rows
Quick look at the top of the dataset to verify structure.

Basic data inspection operation

In [ ]:
df.head()

### 2.3 Preview Last 5 Rows
We inspect the end of the dataframe to check consistent schema.

In [ ]:
df.tail()

### 2.4 Random Dataset Sample
Sampling random rows to check variations in entries.

In [ ]:
df.sample(5)

### 2.5 Dataframe Info & Types
Checking data types and counting non-null values across features.

In [ ]:
df.info()

### 2.6 Descriptive Statistics
Summary stats for numerical attributes to evaluate distributions.

In [ ]:
df.describe()

### 3.1 Calculate Missing Values
Sum of null records per column.

Data cleaning operations

In [ ]:
df.isnull().sum()
#Find missing values

### 3.2 Missing Percentage
Proportion of missing data across attributes.

In [ ]:
(df.isnull().mean() * 100).round(2)
#Find missing values percentage

### 3.3 Deduplication
Identifying and removing duplicate rows.

In [ ]:
#Remove duplicate rows
df = df.drop_duplicates()
df

### 3.4 Numerical Imputation
Filling missing continuous values using column medians.

In [ ]:
#Fill missing numerical values with median
df.fillna(df.median(numeric_only=True), inplace=True)
df

### 3.5 Categorical Imputation
Handling missing fields in categorical features like garage status.

In [ ]:
#Fill missing categorical values
df['has_garage'] = df['has_garage'].fillna('Unknown')
df

### 3.6 Drop Irrelevant Columns
Removing unique identifiers (e.g., ID) that add no statistical value.

In [ ]:
df.drop(columns=['id'], inplace=True)
df
#Remove unnecessary columns

### 3.7 Data Type Conversion
Ensuring structural data types are numerical where appropriate.

In [ ]:
#Convert data types
df['size_sqft'] = pd.to_numeric(
    df['size_sqft'], errors='coerce'
)
df


### 3.8 Diagnostic Quality Check
Let's run automated tests to check for categorical dirty values, duplicate columns, and anomalies in size metrics.

In [ ]:
print("--- Unique values in has_garage ---")
print(df['has_garage'].unique())

print("\n--- Unique values in has_garden ---")
if 'has_garden' in df.columns:
    print(df['has_garden'].unique())

print("\n--- Checking if zipcode and zipcode_dup are identical ---")
if 'zipcode' in df.columns and 'zipcode_dup' in df.columns:
    are_identical = df['zipcode'].equals(df['zipcode_dup'])
    print(f"Are zipcode columns identical? {are_identical}")

print("\n--- Checking for potential extreme outliers in size_sqft ---")
print(df['size_sqft'].describe())

### 3.9 Active Data Cleaning & Standardization
Based on the diagnostic checks above, we will now map categorical variables to clean standard values, drop duplicate zipcode columns, remove extreme size anomalies, and fill any remaining null values.

In [ ]:
# 1. Clean categorical values for 'has_garage'
garage_map = {
    'Ys': 'Yes', 'Ye': 'Yes', 'Yes': 'Yes', 'YES': 'Yes', 'es': 'Yes',
    'No': 'No', 'N': 'No', 'NO': 'No', 'o': 'No', 'Unknown': 'Unknown'
}
df['has_garage'] = df['has_garage'].map(garage_map).fillna('Unknown')

# 2. Clean categorical values for 'has_garden'
garden_map = {
    'Yes': 'Yes', 'Ye': 'Yes', 'Ys': 'Yes', 'YES': 'Yes', 'es': 'Yes', 'o': 'Yes',
    'No': 'No', 'NO': 'No', 'N': 'No'
}
df['has_garden'] = df['has_garden'].map(garden_map).fillna('No')

# 3. Drop duplicate zipcode column
if 'zipcode_dup' in df.columns:
    df.drop(columns=['zipcode_dup'], inplace=True)

# 4. Filter out extreme sqft anomalies (e.g. values > 10,000 sqft)
median_sqft = df['size_sqft'].median()
df.loc[df['size_sqft'] > 10000, 'size_sqft'] = np.nan
df['size_sqft'].fillna(median_sqft, inplace=True)

print("Categorical fields mapped!")
print("Redundant zipcode column removed!")
print("Square footage cleaned and imputed!")

### 4.1 Target Distribution
Visualization of the distribution of housing sale prices.

EDA graphs

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(df['sale_price'].dropna(),
         bins=20, edgecolor='black')

plt.title('Distribution of House Prices')
plt.xlabel('Sale Price')
plt.ylabel('Frequency')

plt.show()

### 4.2 Bedroom Count Distribution
Categorical distribution of the number of bedrooms.

In [ ]:
df['num_bedrooms'].value_counts().sort_index().plot(
    kind='bar', figsize=(8,5)
)

plt.title('Number of Houses by Bedrooms')
plt.xlabel('Bedrooms')
plt.ylabel('Number of Houses')

plt.show()

### 4.3 Garage Status Count
Count plot highlighting classes in the garage status category.

In [ ]:
plt.figure(figsize=(7,5))

sns.countplot(data=df, x='has_garage')

plt.title('Houses with Garage')
plt.xlabel('Garage')
plt.ylabel('Count')

plt.show()

### 4.4 Garage Distribution Pie Chart
Visualizing proportion of different garage status categories.

In [ ]:
df['has_garage'].value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    figsize=(7,7)
)

plt.title('Garage Distribution')
plt.ylabel('')

plt.show()

### 4.5 Sale Price Outliers
Using box plots to detect potential outliers in pricing.

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(y=df['sale_price'])

plt.title('Sale Price Outlier Detection')
plt.ylabel('Sale Price')

plt.show()

### 4.6 Size vs Sale Price
Scatter plot to analyze the relationship between house size and pricing.

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x='size_sqft',
    y='sale_price'
)

plt.title('House Size vs Sale Price')
plt.xlabel('Size in Square Feet')
plt.ylabel('Sale Price')

plt.show()

### 4.7 House Age vs Price Trend
Line plot showcasing pricing trends based on construction year.

In [ ]:
plt.figure(figsize=(8,5))

sns.lineplot(
    data=df,
    x='year_built',
    y='sale_price'
)

plt.title('Year Built vs Sale Price')
plt.xlabel('Year Built')
plt.ylabel('Sale Price')

plt.show()

### 4.8 Correlation Analysis
Heatmap showcasing statistical correlation coefficients between numerical metrics.

In [ ]:
plt.figure(figsize=(10,7))

sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Heatmap')

plt.show()

### 4.9 Multi-Variable Pairplot
Generating pairplots for major numerical features.

In [ ]:
sns.pairplot(
    df[['num_bedrooms',
        'num_bathrooms',
        'year_built',
        'sale_price']].dropna()
)

plt.show()

### 4.10 Price Grouped by Bedrooms
Violin plot displaying price density curves for each bedroom count.

In [ ]:
plt.figure(figsize=(8,5))

sns.violinplot(
    data=df,
    x='num_bedrooms',
    y='sale_price'
)

plt.title('Sale Price by Number of Bedrooms')

plt.show()